In [1]:
import re
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoProcessor
from datasets import load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch
from qwen_vl_utils import process_vision_info
import torch
from transformers import AutoTokenizer, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from datasets import load_dataset

In [2]:
import json
import re
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor

from qwen_vl_utils import process_vision_info  # Qwen 官方提供的 util

class RewardModelDataset(Dataset):
    def __init__(self, json_path, processor, image_root):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.processor = processor
        self.image_root = image_root

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        choices = item["choices"]
        model_out = item["model_out"]

        # 1. 构造消息（符合 chat 模板）
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": Image.open(f"{self.image_root}/{item['image']}").convert("RGB").resize((224, 224))},
                {"type": "text", "text": f"Question: {question}\nChoices: {', '.join(choices)}\nModel Output: {model_out}"},
            ],
        }]

        # 2. 生成 prompt + image/video 结构
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)

        # 3. Processor 封装成模型输入
        inputs = self.processor(
            text=[prompt],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}  # 去掉 batch 维度

        # 4. 加入分数（作为 regression 目标）
        score_str = item.get("model_score", "**[Your Score]: 0.0**")
        score = float(re.search(r'([0-9.]+)', score_str).group(1))
        inputs["score"] = torch.tensor([score])  # ✅ 加上中括号
        return inputs


In [3]:
# 初始化 processor（使用你微调用的同一个）
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# 创建 Dataset 实例
dataset = RewardModelDataset(
    json_path="/root/IC_MLLM_VQA/Score/train_with_score_1_6.json",
    processor=processor,
    image_root="/root/IC_MLLM_VQA/Score/"
)

# 示例读取第一个样本
sample = dataset[2]
print(sample.keys())
print(sample['input_ids'].shape)  # dict: input_ids, attention_mask, pixel_values, score 等
print(sample['score'])  # tensor: 0.0
print(sample['pixel_values'].shape)  # tensor: 图像的像素值
print(sample['attention_mask'].shape)  # tensor: 注意力掩码
print(sample["image_grid_thw"])  # tensor: 图像的网格化像素值



Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'score'])
torch.Size([409])
tensor([0.8000])
torch.Size([256, 1176])
torch.Size([409])
tensor([ 1, 16, 16])


In [4]:
# 设置设备from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoProcessor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 加载模型和处理器
model = AutoModelForVision2Seq.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map={"": device},
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model = PeftModel.from_pretrained(model, "/root/IC_MLLM_VQA/ScienceQA/Lora/qwen2.5vl-Lora1-6_0609/checkpoint-5000")

Using device: cuda


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
import torch.nn as nn
import torch

class RewardModel(nn.Module):
    def __init__(self, qwen_model):
        super().__init__()
        self.backbone = qwen_model
        self.backbone.requires_grad_(False)  # 冻结主模型权重

        # Reward head：从最后一个 token 的 hidden state 映射到 [0, 1] 分数
        self.reward_head = nn.Sequential(
            nn.Linear(self.backbone.config.hidden_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),  # 输出一个分数
            nn.Sigmoid()  # 输出是 0~1 的 score
        )
        self.reward_head = self.reward_head.to(next(qwen_model.parameters()).dtype)

    def forward(self, input_ids, attention_mask, pixel_values, image_grid_thw):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,  # 必传且不可为 None
            output_hidden_states=True,
            return_dict=True,
        )
        last_hidden = outputs.hidden_states[-1]  # [B, S, H]
        pooled = last_hidden[:, -1, :]  # [B, H]
        score = self.reward_head(pooled).squeeze(-1)  # [B]
        
        return score

In [6]:
from transformers import DataCollatorWithPadding
token_collator = DataCollatorWithPadding(tokenizer=processor.tokenizer, return_tensors="pt")

def reward_collate_fn(batch):
    # 图像与结构堆叠
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    image_grid_thw = torch.stack([item["image_grid_thw"] for item in batch])
    scores = torch.tensor(
        [item["score"].item() if isinstance(item["score"], torch.Tensor) else item["score"]
         for item in batch]
    )

    # Token 自动 padding
    token_part = token_collator([
        {k: v for k, v in item.items() if k in ["input_ids", "attention_mask"]}
        for item in batch
    ])

    return {
        **token_part,
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw,
        "score": scores
    }




In [7]:
from torch.utils.data import DataLoader
import torch.nn.functional as F

rm_model = RewardModel(model).to("cuda")
optimizer = torch.optim.SGD(rm_model.reward_head.parameters(), lr=1e-3, momentum=0.9)
train_loader = DataLoader(
    dataset,
    batch_size=12,
    shuffle=True,
    collate_fn=reward_collate_fn
)


In [8]:
for batch in train_loader:
    print(batch.keys())  # 应该有 input_ids, attention_mask, pixel_values, image_grid_thw, score
    print(batch["input_ids"].dtype)  # e.g. [B, seq_len]
    print(batch["pixel_values"].dtype)  # [B, 3, H, W]
    print(batch["score"].dtype)  # [B]
    print(batch["image_grid_thw"].dtype)  # list of B tuples
    break

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'score'])
torch.int64
torch.float32
torch.float32
torch.int64


In [9]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

num_epochs = 3
for epoch in range(num_epochs):
    rm_model.train()
    epoch_loss = 0
    step = 0
    print_every = 20

    # 使用 tqdm 来创建一个进度条，传入 train_loader
    with tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}") as pbar:
        for i, batch in pbar:
            pixel_values = batch["pixel_values"]
            scores = batch["score"]
            if torch.isnan(pixel_values).any() or torch.isinf(pixel_values).any():
                print(f"Batch {i} pixel_values has NaN or Inf")
            if torch.isnan(scores).any() or torch.isinf(scores).any():
                print(f"Batch {i} score has NaN or Inf")
            
            step += 1

            batch = {
                k: (v.to("cuda").half() if isinstance(v, torch.Tensor) and v.dtype == torch.float32 else v.to("cuda"))
                for k, v in batch.items()
            }

            pred = rm_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch["pixel_values"],
                image_grid_thw=batch["image_grid_thw"]
            )

            loss = F.mse_loss(pred, batch["score"])
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()

            # 更新进度条描述
            if step % print_every == 0:
                avg_loss = epoch_loss / step
                pbar.set_postfix(avg_loss=avg_loss)

            # 打印当前 batch 的预测值和真实分数
            if step % print_every == 0:
                print("Pred:", pred.detach().cpu().numpy())
                print("Score:", batch["score"].detach().cpu().numpy())

    print(f"Epoch {epoch + 1} finished - Avg Loss: {epoch_loss / step:.4f}")


Epoch 1:   6%|▌         | 20/363 [00:33<09:31,  1.67s/it, avg_loss=0.0305]

Pred: [0.807  0.804  0.7856 0.7104 0.6943 0.679  0.7007 0.822  0.751  0.7876
 0.785  0.6816]
Score: [0.7 0.8 0.9 0.8 0.8 0.7 0.7 0.8 0.7 0.8 0.7 0.8]


Epoch 1:  11%|█         | 40/363 [01:06<08:14,  1.53s/it, avg_loss=0.025] 

Pred: [0.741  0.609  0.748  0.6694 0.714  0.6685 0.7217 0.7686 0.769  0.6973
 0.7266 0.691 ]
Score: [0.8 0.8 0.7 0.8 0.7 0.8 0.7 0.8 0.8 0.8 0.7 0.5]


Epoch 1:  17%|█▋        | 60/363 [01:39<08:03,  1.59s/it, avg_loss=0.0225]

...

Pred: [0.6846 0.76   0.769  0.771  0.604  0.691  0.737  0.775  0.638  0.7695
 0.7686 0.652 ]
Score: [0.8 0.8 0.8 0.8 0.5 0.8 0.8 0.7 0.5 0.5 0.9 0.7]


Epoch 3:  99%|█████████▉| 360/363 [09:47<00:04,  1.56s/it, avg_loss=0.0125]

Pred: [0.651  0.7563 0.6963 0.6187 0.6265 0.611  0.7183 0.656  0.691  0.744
 0.598  0.7905]
Score: [0.8 0.8 0.7 0.5 0.8 0.5 0.8 0.7 0.7 0.7 0.8 0.9]


Epoch 3: 100%|██████████| 363/363 [09:50<00:00,  1.63s/it, avg_loss=0.0125]

Epoch 3 finished - Avg Loss: 0.0124


In [11]:
# 保存整个 RewardModel
torch.save(rm_model.reward_head.state_dict(), "reward_head_only.pt")
